# Предобработка архивных документов
Пайплайн: выравнивание → разделение на страницы → нормализация освещения

In [ ]:
# !pip install opencv-python-headless numpy matplotlib scikit-image

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from skimage.filters import threshold_sauvola

# ─── Конфигурация ───────────────────────────────────────────────────────────
INPUT_PATH  = 'data/00000014.jpg'   # <- путь к файлу разворота
OUTPUT_DIR  = Path('preproc_results')
OUTPUT_DIR.mkdir(exist_ok=True)

# Параметры (можно менять и перезапускать ячейки)
DESKEW_MAX_ANGLE    = 10    # максимальный угол наклона для поиска (градусы)
SPINE_DARK_THRESH   = 90    # порог яркости для определения полосы переплёта (0–255)
SPINE_MIN_WIDTH     = 20    # минимальная ширина полосы переплёта (пикселей)
CLAHE_CLIP          = 2.0   # clip limit для CLAHE
CLAHE_TILE          = 16    # размер тайла CLAHE

def show(images: list, titles: list, figsize=(18, 8), cmap='gray'):
    """Вспомогательная функция для отображения нескольких изображений."""
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap=cmap if img.ndim == 2 else None)
        ax.set_title(title, fontsize=11)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print('Зависимости загружены ✓')

## Шаг 1 — Загрузка и первичный просмотр

In [ ]:
img_bgr = cv2.imread(INPUT_PATH)
assert img_bgr is not None, f'Файл не найден: {INPUT_PATH}'

img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

h, w = img_gray.shape
print(f'Размер изображения: {w} × {h} px')

show([img_rgb], ['Исходный документ (разворот)'], figsize=(14, 8))

## Шаг 2 — Выравнивание (deskew)

Находим угол наклона через Hough-трансформ на бинаризованном изображении.
Если документ почти ровный — угол будет близок к 0, поворот незаметен.

In [ ]:
def find_skew_angle(gray: np.ndarray, max_angle: float = 10.0) -> float:
    """
    Определяет угол наклона документа через Hough-трансформ.
    Возвращает угол в градусах (положительный = по часовой стрелке).
    """
    # Бинаризация для выделения текстовых линий
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    _, binary = cv2.threshold(blurred, 0, 255,
                               cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Морфология: соединяем буквы в горизонтальные полосы
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (30, 1))
    dilated = cv2.dilate(binary, kernel)

    # Hough для нахождения линий
    lines = cv2.HoughLinesP(
        dilated,
        rho=1, theta=np.pi / 180,
        threshold=100,
        minLineLength=gray.shape[1] // 4,
        maxLineGap=20
    )

    if lines is None:
        print('Hough не нашёл линий, угол = 0')
        return 0.0

    angles = []
    for x1, y1, x2, y2 in lines[:, 0]:
        angle = np.degrees(np.arctan2(y2 - y1, x2 - x1))
        if abs(angle) <= max_angle:   # отфильтровываем вертикали
            angles.append(angle)

    if not angles:
        return 0.0

    # Медиана устойчива к выбросам
    return float(np.median(angles))


def rotate_image(img: np.ndarray, angle: float) -> np.ndarray:
    """Поворачивает изображение на angle градусов относительно центра."""
    h, w = img.shape[:2]
    cx, cy = w // 2, h // 2
    M = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)

    # Вычисляем новые размеры, чтобы не обрезать углы
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    new_w = int(h * sin + w * cos)
    new_h = int(h * cos + w * sin)
    M[0, 2] += new_w / 2 - cx
    M[1, 2] += new_h / 2 - cy

    fill = (255, 255, 255) if img.ndim == 3 else 255
    return cv2.warpAffine(img, M, (new_w, new_h),
                          flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_CONSTANT,
                          borderValue=fill)


angle = find_skew_angle(img_gray, max_angle=DESKEW_MAX_ANGLE)
print(f'Обнаруженный угол наклона: {angle:.2f}°')

img_deskewed_rgb  = rotate_image(img_rgb,  angle)
img_deskewed_gray = rotate_image(img_gray, angle)

show(
    [img_rgb, img_deskewed_rgb],
    [f'До (наклон ~{angle:.1f}°)', 'После выравнивания'],
    figsize=(18, 8)
)

## Шаг 3 — Разделение разворота на две страницы

Ищем тёмную вертикальную полосу переплёта по вертикальному профилю яркости.
Точка разреза — центр этой полосы.

In [ ]:
def find_spine(gray: np.ndarray,
               dark_thresh: int = 80,
               min_width: int = 20,
               search_fraction: float = 0.5) -> int:
    """
    Находит центр полосы переплёта.
    Ищет в центральной search_fraction части изображения.
    Возвращает x-координату центра переплёта.
    """
    h, w = gray.shape
    # Исключаем верхние / нижние 10% (поля, артефакты)
    margin_v = h // 10
    roi = gray[margin_v: h - margin_v, :]

    # Вертикальный профиль средней яркости
    profile = roi.mean(axis=0)

    # Ищем только в центральной части
    cx = w // 2
    half = int(w * search_fraction / 2)
    search = profile[cx - half: cx + half]
    offset = cx - half

    # Тёмные пиксели (переплёт)
    dark_mask = search < dark_thresh

    if not dark_mask.any():
        print('Тёмная полоса не найдена, делим по центру')
        return w // 2

    # Ищем самый широкий непрерывный тёмный отрезок
    best_start, best_len = 0, 0
    cur_start, cur_len = 0, 0
    for i, v in enumerate(dark_mask):
        if v:
            if cur_len == 0:
                cur_start = i
            cur_len += 1
            if cur_len > best_len:
                best_len, best_start = cur_len, cur_start
        else:
            cur_len = 0

    if best_len < min_width:
        print(f'Полоса слишком узкая ({best_len}px), делим по центру')
        return w // 2

    spine_center = offset + best_start + best_len // 2
    print(f'Переплёт: x={offset + best_start}…{offset + best_start + best_len}, '
          f'ширина={best_len}px, центр={spine_center}')
    return spine_center


# Визуализация профиля яркости
h, w = img_deskewed_gray.shape
margin_v = h // 10
profile = img_deskewed_gray[margin_v: h - margin_v, :].mean(axis=0)

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(profile, linewidth=0.8, color='steelblue')
ax.axhline(SPINE_DARK_THRESH, color='red', linewidth=1,
           linestyle='--', label=f'Порог яркости ({SPINE_DARK_THRESH})')
ax.set_xlabel('x (пиксель)')
ax.set_ylabel('Средняя яркость')
ax.set_title('Вертикальный профиль яркости (для поиска переплёта)')
ax.legend()
plt.tight_layout()
plt.show()

spine_x = find_spine(img_deskewed_gray,
                     dark_thresh=SPINE_DARK_THRESH,
                     min_width=SPINE_MIN_WIDTH)

# Показываем линию разреза
preview = img_deskewed_rgb.copy()
cv2.line(preview, (spine_x, 0), (spine_x, h), (255, 0, 0), 3)
show([preview], [f'Линия разреза x={spine_x}'], figsize=(14, 8))

In [ ]:
# Разрезаем (можно скорректировать spine_x вручную если нужно)
# spine_x = 1234   # <- раскомментируй и задай вручную если автодетект неточен

page_left_rgb  = img_deskewed_rgb[:, :spine_x]
page_right_rgb = img_deskewed_rgb[:, spine_x:]
page_left_gray  = img_deskewed_gray[:, :spine_x]
page_right_gray = img_deskewed_gray[:, spine_x:]

show(
    [page_left_rgb, page_right_rgb],
    ['Левая страница', 'Правая страница'],
    figsize=(18, 8)
)

## Шаг 4 — Нормализация освещения (CLAHE)

CLAHE (Contrast Limited Adaptive Histogram Equalization) выравнивает локальную контрастность —
убирает затемнения по краям и у переплёта, не засвечивая светлые области.

In [ ]:
def normalize_lighting(img_rgb: np.ndarray,
                        clip_limit: float = 2.0,
                        tile_size: int = 16) -> np.ndarray:
    """
    Нормализует освещение через CLAHE в пространстве LAB.
    Работает только с каналом L (яркость), цвет не трогает.
    """
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    L, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=clip_limit,
        tileGridSize=(tile_size, tile_size)
    )
    L_eq = clahe.apply(L)

    lab_eq = cv2.merge([L_eq, a, b])
    return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2RGB)


left_norm  = normalize_lighting(page_left_rgb,  CLAHE_CLIP, CLAHE_TILE)
right_norm = normalize_lighting(page_right_rgb, CLAHE_CLIP, CLAHE_TILE)

show(
    [page_left_rgb,  left_norm],
    ['Левая — до', 'Левая — после CLAHE'],
    figsize=(18, 8)
)
show(
    [page_right_rgb, right_norm],
    ['Правая — до', 'Правая — после CLAHE'],
    figsize=(18, 8)
)

## Шаг 5 — Сохранение результатов

In [ ]:
stem = Path(INPUT_PATH).stem

def save(img_rgb: np.ndarray, path: Path):
    cv2.imwrite(str(path), cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    print(f'Сохранено: {path}  ({img_rgb.shape[1]}×{img_rgb.shape[0]})')

save(left_norm,  OUTPUT_DIR / f'{stem}_left.jpg')
save(right_norm, OUTPUT_DIR / f'{stem}_right.jpg')

# Опционально: сохранить выровненный разворот целиком
# save(img_deskewed_rgb, OUTPUT_DIR / f'{stem}_deskewed.jpg')

print('\n✓ Все файлы сохранены в', OUTPUT_DIR.resolve())

## Шаг 6 — Итоговый сравнительный вид

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

pairs = [
    (page_left_rgb,  'Левая — исходная'),
    (page_right_rgb, 'Правая — исходная'),
    (left_norm,      'Левая — обработанная'),
    (right_norm,     'Правая — обработанная'),
]

for ax, (img, title) in zip(axes.flat, pairs):
    ax.imshow(img)
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('Результат предобработки', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()